# 03 Do the forecast gains carry into the portfolios?

Every model is ranked against the naive-timing benchmark `hist_rw` (σ̂²ᵢ,ₜ₊₁ = RVᵢ,ₜ, the Moreira &
Muir portfolio) twice: by forecast accuracy (section 1) and by portfolio performance (section 2).
Section 3 sets the two rankings side by side. Each ranking is shown for the cyclical bucket (pooled)
and for each ETF separately.

This notebook runs no new tests: the forecast results and tests come from e6
(`models/forecasting/evaluation/e6_forecast_summary.ipynb`), the portfolio results and tests
from p2 (`p2_portfolio_eval.ipynb`). The portfolios use the fixed σ̄² (p2, Part C, covers the
expanding σ̄²).
Significance: \* 10%, \*\* 5%, \*\*\* 1%.

## 0 Setup and load

The models, window and settings come from `results/portfolios/run_info.json`, the portfolio results
from p2's tables, and the forecast results from e6's `e6_losses.csv` and `e6_evaluation.csv` (its
`pooled` unit is the cyclical bucket, `Cyclical` here). Run p1, p2 and e6 first; the asserts stop
the notebook if any of them is out of date.

In [ ]:
import sys
import json
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utils.paths import FORECAST_RESULTS_DIR
from utils.portfolio_utils import (
    PORTFOLIO_DIR, EVAL_DIR, SERIES_COLORS, BENCHMARK_COLOR, TEXT_SECONDARY, stars,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

run_info = json.loads((PORTFOLIO_DIR / "run_info.json").read_text())
SELECTED, VARIANTS, TICKERS = run_info["selected"], run_info["variants"], run_info["tickers"]
MAIN_SPEC, BENCHMARK = run_info["main_spec"], run_info["benchmark"]
MODELS = [m for m in VARIANTS if m != BENCHMARK]
FAMILY = pd.Series({m: fam for fam, ms in SELECTED.items() for m in ms}).loc[VARIANTS]
UNITS = TICKERS + ["Cyclical"]

# Portfolio side: p2's tables. Annualised Sharpe ratio and the Ledoit-Wolf test against hist_rw,
# per unit, with the cyclical bucket as the unit "Cyclical"
bucket_tbl = pd.read_csv(EVAL_DIR / f"vs_benchmarks_bucket_{MAIN_SPEC}.csv", index_col="portfolio")
etf_tbl = pd.read_csv(EVAL_DIR / f"vs_benchmarks_per_etf_{MAIN_SPEC}.csv", index_col=["model", "ETF"])
assert set(etf_tbl.index.get_level_values("model")) == set(VARIANTS),     "p2's tables do not match run_info: rerun p2_portfolio_eval.ipynb"
pf = pd.concat([
    etf_tbl[["SR_managed", "SR_diff_vs_RW", "LW_p_vs_RW"]].rename(columns={"SR_managed": "SR"}),
    bucket_tbl.loc[VARIANTS, ["ann_sharpe", "SR_diff_vs_RW", "LW_p_vs_RW"]].rename(columns={"ann_sharpe": "SR"})
    .assign(ETF="Cyclical").set_index("ETF", append=True),
]).rename_axis(["model", "unit"])
sr_tbl = pf["SR"].unstack("unit").loc[VARIANTS, UNITS]

# Forecast side: e6's mean QLIKE, and its tests against hist_rw on the log-RV MSE
e6_losses = (pd.read_csv(FORECAST_RESULTS_DIR / "e6_losses.csv").replace({"ETF": {"pooled": "Cyclical"}})
             .set_index(["model", "ETF"]))
e6_tests = pd.read_csv(FORECAST_RESULTS_DIR / "e6_evaluation.csv").replace({"ETF": {"pooled": "Cyclical"}})
assert set(e6_tests["a"]) == set(MODELS) and (e6_tests["b"] == BENCHMARK).all(),     "e6 results do not match run_info: rerun e6_forecast_summary.ipynb"
q = e6_losses["QLIKE"].unstack("ETF").loc[VARIANTS, UNITS]
p_logmse = (e6_tests[e6_tests["loss"] == "MSE_log"].set_index(["a", "ETF", "test"])["p_value"]
            .unstack("test").rename_axis(["model", "unit"]))   # columns: DM, Clark-West (Elastic Net only)

# Family colours for the plots; the benchmark is grey
FAMILY_COLORS = {fam: SERIES_COLORS[i] for i, fam in enumerate(f for f in SELECTED if f != FAMILY[BENCHMARK])}
color = lambda m: BENCHMARK_COLOR if m == BENCHMARK else FAMILY_COLORS[FAMILY[m]]

print("Models:", VARIANTS)
print(f"Out-of-sample: {run_info['oos_window'][0]} to {run_info['oos_window'][1]} | {MAIN_SPEC} σ̄²")

## 1 Forecast accuracy: ranking by QLIKE

Every model, `hist_rw` included, ranked by mean QLIKE from e6 (rank 1 = lowest QLIKE, the most
accurate). **QLIKE gain %** = (1 − QLIKE / QLIKE of `hist_rw`) × 100, so positive means more accurate
than `hist_rw`.

The significance of the difference against `hist_rw` comes from e6's tests on the **log-RV MSE**,
one-sided (H1: the model is more accurate than `hist_rw`):

- **DM p (log MSE)**: Diebold-Mariano, for every model;
- **CW p (log MSE)**: Clark-West, which e6 runs only for Elastic Net, the family whose models nest
  `hist_rw`; n/a for the others.

In [ ]:
def rank_table(columns, sort_by, ascending):
    """One ranking per unit, hist_rw included: index (unit, rank), rank 1 = best."""
    out = []
    for u in UNITS:
        t = pd.DataFrame(columns(u), index=VARIANTS).rename_axis("model")
        t = t.sort_values(sort_by, ascending=ascending).reset_index()
        out.append(t.assign(unit=u, rank=range(1, len(t) + 1)))
    return pd.concat(out).set_index(["unit", "rank"])


def show_ranking(tbl, units, p_cols, title):
    """Display a ranking with its p-values as 'p stars': '–' for hist_rw, 'n/a' where no test was run."""
    out = tbl.loc[units].copy()
    for col in p_cols:
        out[col] = ["–" if m == BENCHMARK else "n/a" if pd.isna(p) else f"{p:.3f} {stars(p)}".strip()
                    for m, p in zip(out["model"], out[col])]
    if len(units) == 1:
        out = out.droplevel("unit")
    print(title)
    display(out.round(3))


FC_P = ["DM p (log MSE)", "CW p (log MSE)"]
fc_rank = rank_table(lambda u: {
    "family": FAMILY,
    "QLIKE": q[u],
    f"QLIKE gain % vs {BENCHMARK}": (1 - q[u] / q.at[BENCHMARK, u]) * 100,
    FC_P[0]: p_logmse.xs(u, level="unit")["DM"],
    FC_P[1]: p_logmse.xs(u, level="unit")["Clark-West"],
}, sort_by="QLIKE", ascending=True)

show_ranking(fc_rank, ["Cyclical"], FC_P, "Table 1a. Forecast accuracy, cyclical bucket (pooled)")

In [ ]:
show_ranking(fc_rank, TICKERS, FC_P, "Table 1b. Forecast accuracy, per ETF")

## 2 Portfolio performance: ranking by Sharpe ratio

Every model, `hist_rw` included, ranked by the annualised Sharpe ratio of its volatility-managed
portfolio (fixed σ̄², rank 1 = highest). **ΔSR** = SR − SR(`hist_rw`), so positive means a better
portfolio than `hist_rw`. **LW p** is p2's Ledoit-Wolf test of SR = SR(`hist_rw`), a two-sided
p-value from the studentised block bootstrap.

The pooled ranking uses the cyclical bucket. The per-ETF ranking uses each ETF's managed series,
rescaled to the volatility of that ETF's own buy-and-hold.

In [ ]:
PF_P = [f"LW p vs {BENCHMARK}"]
pf_rank = rank_table(lambda u: {
    "family": FAMILY,
    "SR": sr_tbl[u],
    # the ΔSR the Ledoit-Wolf test estimated (within 0.0005 of the plain difference of the SRs)
    f"ΔSR vs {BENCHMARK}": pf.xs(u, level="unit")["SR_diff_vs_RW"].fillna({BENCHMARK: 0.0}),
    PF_P[0]: pf.xs(u, level="unit")["LW_p_vs_RW"],
}, sort_by="SR", ascending=False)

show_ranking(pf_rank, ["Cyclical"], PF_P, f"Table 2a. Portfolio performance, cyclical bucket (pooled, {MAIN_SPEC} σ̄²)")

In [ ]:
show_ranking(pf_rank, TICKERS, PF_P, f"Table 2b. Portfolio performance, per ETF ({MAIN_SPEC} σ̄²)")

## 3 Do better forecasters build better portfolios?

The two pooled rankings side by side: every model, `hist_rw` included, ranked by mean QLIKE (Table
1a) and by the Sharpe ratio of its bucket portfolio (Table 2a). With this few models, Spearman's ρ
is descriptive, not a test.

In [ ]:
ranks = pd.DataFrame({
    "QLIKE": q["Cyclical"],
    "forecast_rank": q["Cyclical"].rank(method="first").astype(int),
    "SR": sr_tbl["Cyclical"],
    "portfolio_rank": sr_tbl["Cyclical"].rank(ascending=False, method="first").astype(int),
})

fig, ax = plt.subplots(figsize=(8, 6))
for m, r in ranks.iterrows():
    bench = m == BENCHMARK
    ax.plot([0, 1], [r["forecast_rank"], r["portfolio_rank"]], color=color(m), linewidth=2 if bench else 1.5,
            linestyle="--" if bench else "-", marker="o", markersize=6)
    ax.text(-0.05, r["forecast_rank"], f"{m}  {r['QLIKE']:.3f}", ha="right", va="center", fontsize=8,
            color=TEXT_SECONDARY)
    ax.text(1.05, r["portfolio_rank"], f"{r['SR']:.3f}  {m}", ha="left", va="center", fontsize=8,
            color=TEXT_SECONDARY)
ax.set_xlim(-0.9, 1.9)
ax.set_ylim(len(ranks) + 0.5, 0.5)
ax.set_xticks([0, 1], ["forecast rank\n(mean QLIKE, e6)", f"portfolio rank\n(Sharpe ratio, {MAIN_SPEC} σ̄²)"])
ax.set_yticks([])
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#b5b4ad")
ax.tick_params(colors=TEXT_SECONDARY, labelsize=9, length=0)
handles = [plt.Line2D([], [], color=BENCHMARK_COLOR, linestyle="--", label=FAMILY[BENCHMARK])]
handles += [plt.Line2D([], [], color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=handles, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12),
          ncol=len(handles))
ax.set_title("Figure 1. Cyclical bucket: forecast rank vs portfolio rank (1 = best)", fontsize=10, loc="left")
fig.tight_layout()
plt.show()

In [ ]:
def spearman(models, unit):
    return (-q.loc[models, unit]).corr(sr_tbl.loc[models, unit], method="spearman")


rho = pd.DataFrame({u: {f"all {len(VARIANTS)} models": spearman(VARIANTS, u),
                        f"without {BENCHMARK}": spearman(MODELS, u)} for u in UNITS})
rank_summary = pd.DataFrame({
    "spearman_rho_all": rho.iloc[0],
    f"spearman_rho_without_{BENCHMARK}": rho.iloc[1],
    "best_forecast_model": q.idxmin(),
    "best_portfolio_model": sr_tbl.idxmax(),
    f"{BENCHMARK}_forecast_rank": q.rank().loc[BENCHMARK].astype(int),
    f"{BENCHMARK}_portfolio_rank": sr_tbl.rank(ascending=False).loc[BENCHMARK].astype(int),
}).rename_axis("unit")
print("Spearman ρ between −QLIKE and Sharpe ratio, per unit (descriptive)")
rho.round(2)

## Save the tables

Written to `results/eval/`: `forecast_ranking.csv` (Tables 1a and 1b), `portfolio_ranking_<spec>.csv`
(Tables 2a and 2b) and `rank_summary_<spec>.csv` (section 3), for the main σ̄². The unit
`Cyclical` is the pooled bucket.

In [ ]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
fc_rank.to_csv(EVAL_DIR / "forecast_ranking.csv")
pf_rank.to_csv(EVAL_DIR / f"portfolio_ranking_{MAIN_SPEC}.csv")
rank_summary.to_csv(EVAL_DIR / f"rank_summary_{MAIN_SPEC}.csv")
print("Saved to", EVAL_DIR.resolve())